# Lecture 4 Demo: Classical ML Refresher Through a Quantum Lens

Computational companion notebook for Lecture 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, load_iris
from sklearn.kernel_approximation import PolynomialCountSketch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

np.random.seed(42)

## Demo 1. Matrix-form linear/logistic baseline

We explicitly compute $z = Xw + b\mathbf{1}$ and then apply a sigmoid.

In [ ]:
X_demo = np.array([
    [1.0, 0.5],
    [0.2, 1.2],
    [1.5, 1.0],
    [0.4, 0.1],
])
w = np.array([1.1, -0.8])
b = -0.1

z = X_demo @ w + b
p = 1.0 / (1.0 + np.exp(-z))

print("z:", np.round(z, 4))
print("sigmoid(z):", np.round(p, 4))

## Demo 2. XOR lifting visualization

We lift XOR from 2D to 3D with $\phi(x_1,x_2)=(x_1,x_2,x_1x_2)$.

In [ ]:
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
], dtype=float)
y_xor = np.array([0, 1, 1, 0])

X_lift = np.column_stack([X_xor[:, 0], X_xor[:, 1], X_xor[:, 0] * X_xor[:, 1]])

fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax1.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="coolwarm", s=80)
ax1.set_title("XOR in 2D")
ax1.set_xlabel("x1")
ax1.set_ylabel("x2")
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.scatter(X_lift[:, 0], X_lift[:, 1], X_lift[:, 2], c=y_xor, cmap="coolwarm", s=80)
ax2.set_title("Lifted XOR in 3D")
ax2.set_xlabel("x1")
ax2.set_ylabel("x2")
ax2.set_zlabel("x1*x2")

plt.tight_layout()
plt.show()

## Demo 3. Gram matrix and kernel-SVM on XOR-style data

In [ ]:
X_moons, y_moons = make_moons(n_samples=220, noise=0.20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42, stratify=y_moons
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svc_lin = SVC(kernel="linear", C=1.0)
svc_rbf = SVC(kernel="rbf", C=1.0, gamma=1.0)
svc_poly = SVC(kernel="poly", C=1.0, degree=3, gamma=1.0, coef0=1.0)

svc_lin.fit(X_train_s, y_train)
svc_rbf.fit(X_train_s, y_train)
svc_poly.fit(X_train_s, y_train)

acc_lin = accuracy_score(y_test, svc_lin.predict(X_test_s))
acc_rbf = accuracy_score(y_test, svc_rbf.predict(X_test_s))
acc_poly = accuracy_score(y_test, svc_poly.predict(X_test_s))

print(f"Linear SVM accuracy: {acc_lin:.3f}")
print(f"RBF SVM accuracy:    {acc_rbf:.3f}")
print(f"Poly SVM accuracy:   {acc_poly:.3f}")

K_poly_train = (X_train_s @ X_train_s.T + 1.0) ** 2
print("Polynomial Gram matrix shape:", K_poly_train.shape)
print("Top-left 4x4 block:\n", np.round(K_poly_train[:4, :4], 3))

## Demo 4. Iris baseline: linear vs RBF

In [ ]:
iris = load_iris()
X_iris = iris.data[:, :2]
y_iris = (iris.target == 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

scaler2 = StandardScaler()
X_tr_s = scaler2.fit_transform(X_tr)
X_te_s = scaler2.transform(X_te)

clf_lin = LogisticRegression(max_iter=2000)
clf_rbf = SVC(kernel="rbf", C=1.0, gamma=1.0)

clf_lin.fit(X_tr_s, y_tr)
clf_rbf.fit(X_tr_s, y_tr)

acc_lr = accuracy_score(y_te, clf_lin.predict(X_te_s))
acc_rbf2 = accuracy_score(y_te, clf_rbf.predict(X_te_s))

print(f"Logistic (linear) accuracy: {acc_lr:.3f}")
print(f"RBF SVM accuracy:            {acc_rbf2:.3f}")